In [1]:
from utils.threebody_solver import PlanetMassModel_WithKeplerModel
from utils import TensorCode_util as tc
import tensorflow as tf
import copy
import numpy as np
from  utils import initvalues_util
import keras
import joblib
import time
import json

%load_ext autoreload
%autoreload 2


In [2]:
model_name = 'DT'
num_bodies = 3

if model_name == 'ANN':
    trained_model = keras.models.load_model('trained_models/ANN_trained.keras')
elif model_name in ['DT', 'RF']:
    trained_model = joblib.load(f'trained_models/{model_name}_trained.joblib')


## Increasing Steps with number of bodies constant

In [3]:
system_scale = 1
tau, n, m, r, v = initvalues_util.initValues(num_bodies)

r_t = [r]
v_t = [v]

r1, v1 = None, None

for k in range(200):
    # print("k ", k)
    rv = np.column_stack([r, v])
    
    rv1 = tc.do_step_wrapper_tfmodel(tau, n, m, rv, model = [])
    r1, v1 = tf.split(rv1, num_or_size_splits=2, axis=-1)
    r1, v1 = r1.numpy(), v1.numpy()

    r_t.append(r1)
    v_t.append(v1)
    r = copy.deepcopy(r1)
    v = copy.deepcopy(v1)

r_t = np.stack(r_t)*system_scale
v_t = np.stack(v_t)*system_scale
rv_t = np.dstack([r_t, v_t])

print("r_t.shape", r_t.shape)

2025-01-13 22:18:02.956802: W tensorflow/core/grappler/optimizers/loop_optimizer.cc:933] Skipping loop optimization for Merge node with control input: StatefulPartitionedCall/while/body/_13/while/cond/then/_148/while/cond/StatefulPartitionedCall/cond/else/_298/cond/cond/else/_527/cond/cond/cond/branch_executed/_871


r_t.shape (201, 3, 3)


In [4]:
tf.config.run_functions_eagerly(True)

In [5]:
m_true = m.copy()
m_initial = np.array(num_bodies*[1])
mask = np.array(num_bodies*[1])
mask[0] = 0
learning_rate = 20.0
n_epochs = 200
batch_size = 1
max_steps = 20

In [ ]:
times_for_convergence = []
learned_masses_errors = []
learned_masses = []

for curr_steps in range(max_steps):

    print("Current steps ", curr_steps)
    dataset = np.dstack((rv_t[np.newaxis, 0, ...], rv_t[np.newaxis, 1 + curr_steps, ...]))
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)

    pmm = PlanetMassModel_WithKeplerModel(time_step=tau, initial_masses=m_initial, num_of_steps=1+curr_steps, mass_penalty=0, trained_model=[], mask = mask)
    pmm.compile(loss=pmm.loss_fn, optimizer=optimizer)

    start = time.time()
    historian = pmm.fit(dataset, epochs=n_epochs, batch_size=batch_size, verbose=0)
    elapsed_seconds = time.time() - start

    print(np.mean((m - pmm.masses) ** 2))
    times_for_convergence.append(elapsed_seconds)
    learned_masses_errors.append(np.mean((m - pmm.masses) ** 2))
    learned_masses.append(pmm.masses)


In [ ]:
times_for_convergence_trained_model = []
learned_masses_errors_trained_model = []
learned_masses_trained_model = []

for curr_steps in range(max_steps):

    print("Current steps ", curr_steps)
    dataset = np.dstack(
        (rv_t[np.newaxis, 0, ...], rv_t[np.newaxis, 1 + curr_steps, ...]))
    
    optimizer = keras.optimizers.Adam(learning_rate=5.0)

    pmm = PlanetMassModel_WithKeplerModel(time_step=tau, initial_masses=m_initial, num_of_steps=1 +
                                          curr_steps, mass_penalty=0, trained_model=[model_name, trained_model], mask=mask)
    pmm.compile(loss=pmm.loss_fn, optimizer=optimizer)

    start = time.time()
    historian = pmm.fit(dataset, epochs=n_epochs, batch_size=batch_size, verbose=0)
    elapsed_seconds = time.time() - start

    times_for_convergence_trained_model.append(elapsed_seconds)
    learned_masses_errors_trained_model.append(np.mean((m - pmm.masses) ** 2))
    learned_masses_trained_model.append(pmm.masses)

In [13]:
results = {"kepler_solver": [learned_masses_errors, times_for_convergence], f"{model_name}": [learned_masses_errors_trained_model, times_for_convergence_trained_model]}

In [ ]:
with open(f"learning_masses_results/{model_name}_results_{num_bodies}_bodies.json", "w") as outfile: 
    json.dump(results, outfile)